# Python Functions — A Complete Guide

> **Series:** Learn the Basics | **Previous:** Loops

Functions are the fundamental building block of reusable, maintainable code.
This notebook covers everything from basic definitions to closures,
decorators, and the `functools` toolkit.

---

## Table of Contents
1. [Defining Functions](#1.-Defining-Functions)
2. [Arguments & Parameters](#2.-Arguments-&-Parameters)
3. [*args and **kwargs](#3.-*args-and-**kwargs)
4. [Parameter Ordering Rules](#4.-Parameter-Ordering-Rules)
5. [Return Values](#5.-Return-Values)
6. [Scope — The LEGB Rule](#6.-Scope-—-The-LEGB-Rule)
7. [Docstrings & Annotations](#7.-Docstrings-&-Annotations)
8. [Lambda Functions](#8.-Lambda-Functions)
9. [Higher-order Functions](#9.-Higher-order-Functions)
10. [Closures](#10.-Closures)
11. [Decorators](#11.-Decorators)
12. [Recursion](#12.-Recursion)
13. [functools Module](#13.-functools-Module)
14. [Quick Reference Card](#14.-Quick-Reference-Card)


---
## 1. Defining Functions

```python
def function_name(parameters):
    """Optional docstring."""
    # body
    return value
```

- `def` creates a **function object** and binds it to the name.
- The body runs only when the function is **called**.
- Without an explicit `return`, the function returns `None`.
- Functions are **first-class objects** — they can be passed around like any value.


In [ ]:
# Simplest function
def greet():
    print("Hello, World!")

greet()    # call it

# Functions are objects
print(type(greet))        # <class 'function'>
print(greet.__name__)     # greet


In [ ]:
# Function with a parameter and return value
def square(x):
    return x ** 2

print(square(5))     # 25
print(square(1.5))   # 2.25

# Assign to variable, pass to another function
fn = square
print(fn(7))         # 49 — same function, different name
results = list(map(square, range(1, 6)))
print(results)       # [1, 4, 9, 16, 25]


In [ ]:
# Function with no explicit return returns None
def say_hi(name):
    print(f"Hi, {name}!")

result = say_hi("Alice")
print(f"Return value: {result!r}")   # None

# Early return
def safe_divide(a, b):
    if b == 0:
        return None    # early exit
    return a / b

print(safe_divide(10, 2))    # 5.0
print(safe_divide(10, 0))    # None


In [ ]:
# Nested functions
def outer():
    def inner():
        return "I am inner"
    return inner()   # call inner from outer

print(outer())

# Functions can return other functions
def make_greeter(greeting):
    def greeter(name):
        return f"{greeting}, {name}!"
    return greeter          # return the function object

hello = make_greeter("Hello")
howdy = make_greeter("Howdy")
print(hello("Alice"))
print(howdy("Bob"))


---
## 2. Arguments & Parameters

| Term | Description |
|------|-------------|
| **Parameter** | Variable in the function definition |
| **Argument** | Actual value passed during the call |
| **Positional** | Matched left-to-right by position |
| **Keyword** | Matched by name: `f(x=1)` |
| **Default** | Parameter with a preset value |


In [ ]:
# Positional arguments
def add(a, b):
    return a + b

print(add(3, 4))        # positional
print(add(b=4, a=3))    # keyword (order doesn't matter)
print(add(3, b=4))      # mixed: positional then keyword


In [ ]:
# Default parameter values
def power(base, exponent=2):
    return base ** exponent

print(power(3))       # 3^2 = 9  (uses default)
print(power(3, 3))    # 3^3 = 27 (overrides default)
print(power(2, 10))   # 2^10 = 1024

# Real-world example: connect() with sensible defaults
def connect(host, port=5432, timeout=30, ssl=True):
    return f"Connecting to {host}:{port} (timeout={timeout}s, ssl={ssl})"

print(connect("db.example.com"))
print(connect("db.example.com", port=3306, ssl=False))


In [ ]:
# GOTCHA: Mutable default arguments — the classic Python trap!
def bad_append(item, lst=[]):   # list created ONCE at definition time
    lst.append(item)
    return lst

print(bad_append(1))   # [1]    (looks correct)
print(bad_append(2))   # [1, 2] (BUG! same list reused)
print(bad_append(3))   # [1, 2, 3] (still growing!)

print()
# CORRECT: use None as sentinel, create fresh object inside
def good_append(item, lst=None):
    if lst is None:
        lst = []           # new list each call
    lst.append(item)
    return lst

print(good_append(1))   # [1]
print(good_append(2))   # [2]  (fresh list each time)


In [ ]:
# Passing mutable vs immutable objects
def try_modify_int(x):
    x += 100      # rebinds local x, original unchanged
    print(f"  inside: x={x}")

def try_modify_list(lst):
    lst.append(99)  # mutates the SAME object
    print(f"  inside: lst={lst}")

n = 5
try_modify_int(n)
print(f"outside: n={n}")   # unchanged

print()
data = [1, 2, 3]
try_modify_list(data)
print(f"outside: data={data}")   # CHANGED!


---
## 3. *args and **kwargs

| Syntax | What it captures | Type inside function |
|--------|-----------------|---------------------|
| `*args` | Any number of positional arguments | `tuple` |
| `**kwargs` | Any number of keyword arguments | `dict` |

The names `args` and `kwargs` are convention — you can use any valid name.


In [ ]:
# *args — variadic positional arguments
def add_all(*args):
    print(f"args: {args}  (type: {type(args).__name__})")
    return sum(args)

print(add_all(1, 2, 3))
print(add_all(10, 20, 30, 40, 50))
print(add_all())         # zero args is fine


In [ ]:
# **kwargs — variadic keyword arguments
def show_info(**kwargs):
    print(f"kwargs: {kwargs}  (type: {type(kwargs).__name__})")
    for key, val in kwargs.items():
        print(f"  {key} = {val}")

show_info(name="Alice", age=30, city="Paris")


In [ ]:
# Mixing regular, *args, and **kwargs
def describe(title, *items, **options):
    print(f"Title  : {title}")
    print(f"Items  : {items}")
    print(f"Options: {options}")

describe(
    "Shopping List",
    "apple", "banana", "cherry",
    sort=True, bold=False,
)


In [ ]:
# Unpacking into function calls with * and **
def volume(length, width, height):
    return length * width * height

dims_list  = [3, 4, 5]
dims_dict  = {"length": 3, "width": 4, "height": 5}

print(volume(*dims_list))    # unpack list as positional
print(volume(**dims_dict))   # unpack dict as keyword

# Combine both
coords = (10, 20)
meta   = {"height": 30}
print(volume(*coords, **meta))


In [ ]:
# Practical: forwarding args to another function
def log_call(func, *args, **kwargs):
    print(f"Calling {func.__name__}({args}, {kwargs})")
    result = func(*args, **kwargs)
    print(f"  -> {result}")
    return result

log_call(max, 3, 7, 2, 9, 1)
log_call(sorted, [5, 1, 3], reverse=True)


---
## 4. Parameter Ordering Rules

Python 3 allows fine-grained control over how arguments must be passed:

```python
def f(pos_only, /, normal, *, kw_only):
    ...
```

| Marker | Meaning |
|--------|---------|
| `/` | Everything **before** `/` is positional-only (Python 3.8+) |
| `*` | Everything **after** `*` is keyword-only |
| `*args` | Also acts as a keyword-only separator for what follows |

Full order: `positional_only / regular *args keyword_only **kwargs`


In [ ]:
# Keyword-only parameters (after *)
def create_user(name, *, admin=False, active=True):
    return {"name": name, "admin": admin, "active": active}

print(create_user("Alice"))                      # defaults
print(create_user("Bob", admin=True))            # keyword

# This would fail — admin cannot be passed positionally:
try:
    create_user("Carol", True)
except TypeError as e:
    print(f"TypeError: {e}")


In [ ]:
# Positional-only parameters (before /)
def circle_area(radius, /):
    import math
    return math.pi * radius ** 2

print(f"{circle_area(5):.4f}")

# This would fail — radius cannot be keyword:
try:
    circle_area(radius=5)
except TypeError as e:
    print(f"TypeError: {e}")


In [ ]:
# Full signature using all markers
def full_example(a, b, /, c, d, *, e, f):
    return f"a={a} b={b} c={c} d={d} e={e} f={f}"

# a, b: positional-only
# c, d: positional or keyword
# e, f: keyword-only
print(full_example(1, 2, 3, d=4, e=5, f=6))
print(full_example(1, 2, c=3, d=4, e=5, f=6))


---
## 5. Return Values

- A function can return **any Python object** (including functions!).
- `return` with no value, or no `return`, returns `None`.
- Return a **tuple** to return multiple values (Python unpacks it automatically).


In [ ]:
# Returning multiple values (via tuple)
def min_max(numbers):
    return min(numbers), max(numbers)    # returns a tuple

low, high = min_max([3, 1, 4, 1, 5, 9, 2, 6])
print(f"min={low}, max={high}")

# Or capture as tuple
result = min_max([10, 20, 30])
print(f"type: {type(result)}, value: {result}")


In [ ]:
# Returning None explicitly vs implicitly
def process(data):
    if not data:
        return        # explicit None return (early exit)
    return [x * 2 for x in data]

print(process([]))         # None
print(process([1, 2, 3]))  # [2, 4, 6]

# Returning different types based on input (generally avoid)
def flexible(x):
    if isinstance(x, list): return sum(x)
    if isinstance(x, str):  return x.upper()
    return x

print(flexible([1, 2, 3]))   # 6
print(flexible("hello"))     # HELLO
print(flexible(42))          # 42


In [ ]:
# Named tuple for structured returns (more readable than plain tuples)
from collections import namedtuple

Stats = namedtuple("Stats", ["mean", "median", "std"])

def compute_stats(data):
    import statistics
    return Stats(
        mean   = statistics.mean(data),
        median = statistics.median(data),
        std    = statistics.stdev(data),
    )

data = [4, 8, 15, 16, 23, 42]
s = compute_stats(data)
print(f"mean   = {s.mean:.2f}")
print(f"median = {s.median}")
print(f"std    = {s.std:.2f}")

mean, median, std = s    # still unpackable
print(f"unpacked: {mean:.2f}, {median}, {std:.2f}")


---
## 6. Scope — The LEGB Rule

Python resolves names in this order:

| Letter | Scope | Description |
|--------|-------|-------------|
| **L** | Local | Inside the current function |
| **E** | Enclosing | Enclosing function(s) — for closures |
| **G** | Global | Module-level names |
| **B** | Built-in | Python's built-ins (`len`, `print`, etc.) |

Python looks up each scope in order and uses the **first match** it finds.


In [ ]:
x = "global"      # G — global scope

def outer():
    x = "enclosing"  # E — enclosing scope

    def inner():
        x = "local"  # L — local scope
        print(f"inner  sees: x = {x!r}")

    inner()
    print(f"outer  sees: x = {x!r}")

outer()
print(f"global sees: x = {x!r}")


In [ ]:
# global keyword — modify a global variable from inside a function
counter = 0

def increment():
    global counter       # declare intent to use global
    counter += 1

increment()
increment()
increment()
print(f"counter = {counter}")   # 3

# Better approach: return new value instead of mutating global
def increment_pure(n):
    return n + 1

c = 0
c = increment_pure(c)
c = increment_pure(c)
print(f"pure counter = {c}")


In [ ]:
# nonlocal keyword — modify enclosing scope variable
def make_counter(start=0):
    count = start

    def increment(step=1):
        nonlocal count     # modify the enclosing 'count'
        count += step
        return count

    def reset():
        nonlocal count
        count = start

    def value():
        return count       # just read (no nonlocal needed)

    return increment, reset, value

inc, rst, val = make_counter(10)
print(inc())      # 11
print(inc(5))     # 16
print(val())      # 16
rst()
print(val())      # 10 (reset to start)


In [ ]:
# Built-in scope — Python's built-ins are always available
print(type(len))          # <class 'builtin_function_or_method'>

# Shadowing a built-in (avoid!)
def bad_example():
    list = [1, 2, 3]   # shadows built-in 'list'
    try:
        list(range(5))  # TypeError: 'list' object is not callable
    except TypeError as e:
        print(f"Error: {e}")
    return list

result = bad_example()
print(result)
print(list(range(5)))   # built-in still fine outside function


---
## 7. Docstrings & Annotations

### Docstrings
A string literal as the **first statement** of a function becomes its docstring
(`__doc__`). Accessible via `help()` and IDEs.

### Type Annotations (Python 3.5+)
Annotations describe the **expected types** of parameters and return values.
They are **not enforced at runtime** — they serve as documentation and enable
static analysis tools like `mypy`.


In [ ]:
# Docstring styles

# One-liner
def add(a, b):
    """Return the sum of a and b."""
    return a + b

# Google style (multi-line)
def divide(dividend, divisor):
    """
    Divide dividend by divisor.

    Args:
        dividend: The number to be divided.
        divisor:  The number to divide by.

    Returns:
        The quotient as a float.

    Raises:
        ZeroDivisionError: If divisor is zero.
    """
    if divisor == 0:
        raise ZeroDivisionError("divisor cannot be zero")
    return dividend / divisor

print(add.__doc__)
print()
help(divide)


In [ ]:
# Type annotations
def greet(name: str, times: int = 1) -> str:
    """Return a greeting repeated 'times' times."""
    return (f"Hello, {name}! " * times).strip()

print(greet("Alice"))
print(greet("Bob", 3))

# Inspect annotations
print(f"annotations: {greet.__annotations__}")


In [ ]:
# Complex type annotations (from typing module)
from typing import Optional, Union, List, Dict, Tuple, Callable, Any

def process_data(
    data: List[int],
    multiplier: Union[int, float] = 1.0,
    label: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Process a list of integers.

    Args:
        data:       Input list of integers.
        multiplier: Scale factor (int or float).
        label:      Optional label for the result.

    Returns:
        Dictionary with scaled values and stats.
    """
    scaled = [x * multiplier for x in data]
    return {
        "label":  label or "unlabeled",
        "scaled": scaled,
        "total":  sum(scaled),
        "count":  len(scaled),
    }

result = process_data([1, 2, 3, 4, 5], multiplier=2.5, label="test")
for k, v in result.items():
    print(f"  {k}: {v}")


---
## 8. Lambda Functions

A **lambda** is an anonymous single-expression function.

```python
lambda parameters: expression
```

- Limited to **one expression** (no statements, no multi-line body).
- Returns the value of the expression implicitly.
- Best used as **short inline callbacks** (e.g. `key=` argument).
- For anything more complex, use a named `def`.


In [ ]:
# Lambda basics
double = lambda x: x * 2
add    = lambda a, b: a + b
greet  = lambda name="World": f"Hello, {name}!"

print(double(5))
print(add(3, 4))
print(greet())
print(greet("Alice"))

# Equivalent def
def double_def(x):
    return x * 2

print(double(7) == double_def(7))


In [ ]:
# Lambda as key function
students = [
    {"name": "Alice", "gpa": 3.8},
    {"name": "Bob",   "gpa": 3.2},
    {"name": "Carol", "gpa": 3.9},
    {"name": "Dave",  "gpa": 3.5},
]

# Sort by GPA descending
ranked = sorted(students, key=lambda s: s["gpa"], reverse=True)
for i, s in enumerate(ranked, 1):
    print(f"  {i}. {s['name']} — GPA {s['gpa']}")

# Sort by last name
names = ["Alice Smith", "Bob Jones", "Carol White", "Dave Brown"]
by_last = sorted(names, key=lambda n: n.split()[-1])
print(f"\nBy last name: {by_last}")


In [ ]:
# Lambda in higher-order functions
nums = [-3, -1, 0, 2, 4, 6, -5]

positive  = list(filter(lambda x: x > 0, nums))
doubled   = list(map(lambda x: x * 2, nums))
abs_vals  = list(map(abs, nums))   # built-in also works

print(f"positive: {positive}")
print(f"doubled : {doubled}")
print(f"abs_vals: {abs_vals}")


In [ ]:
# When NOT to use lambda

# Bad: complex logic is unreadable as lambda
# f = lambda x: x**2 if x > 0 else -x if x < -10 else 0

# Good: use a named function
def transform(x):
    if x > 0:    return x ** 2
    if x < -10:  return -x
    return 0

print([transform(x) for x in [-15, -5, 0, 3, 7]])

# Bad: assigning lambda to a name (just use def)
# sq = lambda x: x**2     # PEP8 discourages this
def sq(x): return x**2   # preferred

print(sq(9))


---
## 9. Higher-order Functions

A **higher-order function** either:
- **Takes** a function as an argument, or
- **Returns** a function.

| Built-in | Description |
|----------|-------------|
| `map(fn, it)` | Apply fn to each element |
| `filter(fn, it)` | Keep elements where fn is True |
| `sorted(it, key=fn)` | Sort with custom key |
| `min/max(it, key=fn)` | Find extreme with custom key |
| `functools.reduce(fn, it)` | Accumulate with binary function |


In [ ]:
# map() — apply a function to every element
nums = [1, 4, 9, 16, 25]

roots  = list(map(lambda x: x**0.5, nums))
strs   = list(map(str, nums))

print(f"roots : {roots}")
print(f"strs  : {strs}")

# map with multiple iterables
a = [1, 2, 3]
b = [10, 20, 30]
sums = list(map(lambda x, y: x + y, a, b))
print(f"pair sums: {sums}")

# Modern preference: list comprehension (usually clearer)
sums_comp = [x + y for x, y in zip(a, b)]
print(f"comp sums: {sums_comp}")


In [ ]:
# filter() — keep elements that satisfy a predicate
words = ["apple", "", "banana", None, "cherry", "", "date"]

non_empty = list(filter(None, words))     # filter(None, ...) removes falsy values
long_words = list(filter(lambda w: w and len(w) > 5, words))

print(f"non_empty  : {non_empty}")
print(f"long_words : {long_words}")

# Equivalent comprehensions
non_empty_c  = [w for w in words if w]
long_words_c = [w for w in words if w and len(w) > 5]
print(f"comp match : {non_empty == non_empty_c and long_words == long_words_c}")


In [ ]:
# sorted() / min() / max() with key functions
words = ["banana", "Apple", "cherry", "DATE", "elderberry"]

print(f"default sort  : {sorted(words)}")
print(f"case-insensitive: {sorted(words, key=str.lower)}")
print(f"by length     : {sorted(words, key=len)}")
print(f"by last char  : {sorted(words, key=lambda w: w[-1].lower())}")

nums = [3, -7, 1, -2, 5, -9]
print(f"min by abs    : {min(nums, key=abs)}")
print(f"max by abs    : {max(nums, key=abs)}")


In [ ]:
# Writing your own higher-order functions
def apply_twice(fn, x):
    return fn(fn(x))

print(apply_twice(lambda x: x + 3, 10))  # (10+3)+3 = 16
print(apply_twice(str.upper, "hello"))   # HELLO (already upper, no change)

def compose(*funcs):
    """Compose functions right-to-left: compose(f, g)(x) == f(g(x))."""
    def composed(x):
        for fn in reversed(funcs):
            x = fn(x)
        return x
    return composed

pipeline = compose(
    lambda s: s + "!",
    str.title,
    str.strip,
)
print(pipeline("  hello world  "))   # Hello World!


---
## 10. Closures

A **closure** is a function that **remembers the variables from its enclosing scope**
even after that scope has finished executing.

```python
def outer(x):
    def inner():      # inner 'closes over' x
        return x
    return inner      # return the function
```

Closures are the foundation of **decorators**, **factories**, and **callbacks**.


In [ ]:
# Basic closure
def make_multiplier(n):
    def multiplier(x):     # closes over 'n'
        return x * n
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)

print(f"double(5) = {double(5)}")
print(f"triple(5) = {triple(5)}")

# Each closure has its own copy of 'n'
print(f"double.__closure__[0].cell_contents = {double.__closure__[0].cell_contents}")
print(f"triple.__closure__[0].cell_contents = {triple.__closure__[0].cell_contents}")


In [ ]:
# Classic closure gotcha: late binding in loops

# BUG: all functions share the same 'i' variable (late binding)
bad_funcs = []
for i in range(5):
    bad_funcs.append(lambda x: x * i)

print("Bad (all use i=4):", [f(1) for f in bad_funcs])   # [4,4,4,4,4]

# FIX 1: capture at definition time with default argument
good_funcs1 = []
for i in range(5):
    good_funcs1.append(lambda x, i=i: x * i)   # i=i captures current value

print("Fix1 (default arg):", [f(1) for f in good_funcs1])  # [0,1,2,3,4]

# FIX 2: factory function
def make_fn(i):
    return lambda x: x * i

good_funcs2 = [make_fn(i) for i in range(5)]
print("Fix2 (factory):    ", [f(1) for f in good_funcs2])  # [0,1,2,3,4]


In [ ]:
# Closure as a stateful object (alternative to a class)
def make_accumulator(initial=0):
    total = initial

    def add(n):
        nonlocal total
        total += n
        return total

    def reset():
        nonlocal total
        total = initial

    add.reset = reset   # attach reset as an attribute
    return add

acc = make_accumulator(100)
print(acc(10))   # 110
print(acc(20))   # 130
print(acc(5))    # 135
acc.reset()
print(acc(1))    # 101


---
## 11. Decorators

A **decorator** is a function that wraps another function to modify or
extend its behaviour — without changing its source code.

```python
@decorator
def my_function():
    ...
# equivalent to: my_function = decorator(my_function)
```


In [ ]:
# Building a decorator from scratch
import time

def timer(func):
    """Measure execution time of a function."""
    def wrapper(*args, **kwargs):
        start  = time.perf_counter()
        result = func(*args, **kwargs)
        end    = time.perf_counter()
        print(f"  [{func.__name__}] took {end - start:.6f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

result = slow_sum(1_000_000)
print(f"  result = {result}")


In [ ]:
# Preserving function metadata with functools.wraps
from functools import wraps

def timer_proper(func):
    @wraps(func)             # copies __name__, __doc__, __annotations__
    def wrapper(*args, **kwargs):
        start  = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"  [{func.__name__}] {elapsed:.6f}s")
        return result
    return wrapper

@timer_proper
def compute(n):
    """Compute sum of squares."""
    return sum(i**2 for i in range(n))

compute(100_000)
print(f"name : {compute.__name__}")   # compute (not wrapper)
print(f"doc  : {compute.__doc__}")


In [ ]:
# Decorator with arguments (decorator factory)
from functools import wraps

def retry(max_attempts=3, delay=0.1):
    """Retry a function on exception."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"  Attempt {attempt} failed: {e}")
                    if attempt == max_attempts:
                        raise
                    time.sleep(delay)
        return wrapper
    return decorator

import random; random.seed(0)

@retry(max_attempts=4, delay=0)
def flaky_operation():
    if random.random() < 0.6:    # fails 60% of the time
        raise ValueError("Random failure")
    return "Success!"

print(flaky_operation())


In [ ]:
# Stacking decorators
from functools import wraps

def bold(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return "<b>" + func(*args, **kwargs) + "</b>"
    return wrapper

def italic(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return "<i>" + func(*args, **kwargs) + "</i>"
    return wrapper

@bold
@italic          # applied bottom-up: italic first, then bold
def greet(name):
    return f"Hello, {name}!"

print(greet("Alice"))   # <b><i>Hello, Alice!</i></b>

# Manual equivalent:
# greet = bold(italic(greet))


In [ ]:
# Class-based decorator
class CallCount:
    """Count how many times a function has been called."""

    def __init__(self, func):
        wraps(func)(self)
        self.func  = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        return self.func(*args, **kwargs)

@CallCount
def say_hello(name):
    return f"Hello, {name}!"

print(say_hello("Alice"))
print(say_hello("Bob"))
print(say_hello("Carol"))
print(f"Called {say_hello.count} times")


---
## 12. Recursion

A function is **recursive** when it calls itself. Every recursive solution
needs:
1. A **base case** — stops the recursion.
2. A **recursive case** — reduces the problem toward the base case.

Python's default recursion limit is **1000** (`sys.getrecursionlimit()`).


In [ ]:
# Factorial — classic recursion
def factorial(n):
    if n <= 1:          # base case
        return 1
    return n * factorial(n - 1)   # recursive case

for i in range(11):
    print(f"  {i}! = {factorial(i)}")


In [ ]:
# Fibonacci — naive recursion (exponential time)
def fib_naive(n):
    if n <= 1: return n
    return fib_naive(n - 1) + fib_naive(n - 2)

# With memoization (linear time)
from functools import lru_cache

@lru_cache(maxsize=None)
def fib_memo(n):
    if n <= 1: return n
    return fib_memo(n - 1) + fib_memo(n - 2)

import time
t0 = time.perf_counter(); fib_naive(35); t1 = time.perf_counter()
t2 = time.perf_counter(); fib_memo(35);  t3 = time.perf_counter()
print(f"naive fib(35): {t1-t0:.4f}s")
print(f"memo  fib(35): {t3-t2:.6f}s")
print(f"fib(100) = {fib_memo(100)}")


In [ ]:
# Tree recursion — traverse nested structures
def flatten(data):
    """Recursively flatten a nested list of any depth."""
    result = []
    for item in data:
        if isinstance(item, list):
            result.extend(flatten(item))   # recurse into sublists
        else:
            result.append(item)
    return result

deep = [1, [2, 3], [4, [5, [6, 7]]], 8]
print(flatten(deep))

# Count nodes in a nested dict
def count_nodes(d):
    if not isinstance(d, dict):
        return 1
    return sum(count_nodes(v) for v in d.values())

tree = {"a": {"b": 1, "c": {"d": 2, "e": 3}}, "f": 4}
print(f"nodes: {count_nodes(tree)}")


In [ ]:
# Tail-call optimisation: Python doesn't have it,
# but we can convert to iterative manually

def factorial_iter(n):
    """Iterative factorial — no stack overflow risk."""
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result

print(factorial_iter(20))

# Check recursion limit
import sys
print(f"recursion limit: {sys.getrecursionlimit()}")


---
## 13. functools Module

The `functools` module provides higher-order function utilities.

| Function | Description |
|----------|-------------|
| `wraps(fn)` | Preserve metadata when writing decorators |
| `lru_cache(maxsize)` | Memoize with LRU cache |
| `cache` | Unbounded memoization (3.9+) |
| `partial(fn, ...)` | Pre-fill arguments |
| `reduce(fn, it)` | Left-fold / accumulate |
| `total_ordering` | Auto-generate comparison methods from `__eq__` + one other |
| `singledispatch` | Single-dispatch generic functions |


In [ ]:
from functools import lru_cache, cache

# lru_cache — cache the N most recent results
@lru_cache(maxsize=128)
def expensive(n):
    import time; time.sleep(0.001)   # simulate work
    return n ** 2

import time
t0 = time.perf_counter()
for _ in range(10): expensive(42)   # first call is slow, rest are cached
print(f"10 calls: {time.perf_counter()-t0:.4f}s")
print(f"cache info: {expensive.cache_info()}")

expensive.cache_clear()             # invalidate cache
print(f"after clear: {expensive.cache_info()}")


In [ ]:
from functools import partial

# partial() — create a new function with some args pre-filled
def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)
cube   = partial(power, exponent=3)

print(f"square(5) = {square(5)}")
print(f"cube(4)   = {cube(4)}")

# Practical: pre-configure a print function
error_print = partial(print, end="\n", sep=" | ", flush=True)
error_print("ERROR", "File not found", "/etc/missing.conf")

# partial with positional args
from functools import partial
double = partial(map, lambda x: x * 2)
print(list(double([1, 2, 3, 4])))


In [ ]:
from functools import reduce
import operator

# reduce(fn, iterable) — apply fn cumulatively
nums = [1, 2, 3, 4, 5]

total   = reduce(operator.add, nums)           # sum
product = reduce(operator.mul, nums)           # product
maximum = reduce(lambda a, b: a if a > b else b, nums)  # max

print(f"sum     : {total}")
print(f"product : {product}")
print(f"max     : {maximum}")

# With initial value
result = reduce(lambda acc, x: acc + [x**2], nums, [])
print(f"squares : {result}")


In [ ]:
from functools import singledispatch

# singledispatch — different behaviour based on argument type
@singledispatch
def process(arg):
    raise TypeError(f"Unsupported type: {type(arg).__name__}")

@process.register(int)
def _(arg):
    return f"int: {arg ** 2}"

@process.register(str)
def _(arg):
    return f"str: {arg.upper()}"

@process.register(list)
def _(arg):
    return f"list: {sorted(arg)}"

print(process(5))
print(process("hello"))
print(process([3, 1, 2]))
try:
    process(3.14)
except TypeError as e:
    print(f"TypeError: {e}")


---
## 14. Quick Reference Card


In [ ]:
# ==================================================================
# PYTHON FUNCTIONS – QUICK REFERENCE
# ==================================================================
from functools import wraps, lru_cache, partial, reduce
import operator

# --- Basic definition ---
def add(a, b=0): return a + b
print(add(3), add(3, 4))

# --- *args / **kwargs ---
def variadic(*args, **kwargs): return args, kwargs
print(variadic(1, 2, x=3))

# --- Keyword-only / positional-only ---
def strict(pos_only, /, regular, *, kw_only):
    return pos_only + regular + kw_only
print(strict(1, 2, kw_only=3))

# --- Multiple return ---
def minmax(lst): return min(lst), max(lst)
lo, hi = minmax([3, 1, 4, 1, 5])
print(f"lo={lo}, hi={hi}")

# --- Lambda ---
sq = lambda x: x**2
print(sorted([3,1,4,1,5], key=sq))

# --- Higher-order ---
print(list(map(sq, [1,2,3,4])))
print(list(filter(lambda x: x%2==0, range(10))))

# --- Closure ---
def multiplier(n): return lambda x: x * n
triple = multiplier(3)
print(triple(7))

# --- Decorator ---
def uppercase(fn):
    @wraps(fn)
    def wrapper(*a, **kw): return fn(*a, **kw).upper()
    return wrapper

@uppercase
def hello(name): return f"hello, {name}"
print(hello("world"))

# --- lru_cache ---
@lru_cache(maxsize=32)
def fib(n): return n if n<2 else fib(n-1)+fib(n-2)
print(fib(30))

# --- partial ---
double = partial(map, lambda x: x*2)
print(list(double([1,2,3])))

# --- reduce ---
print(reduce(operator.mul, range(1,6)))  # 5! = 120

# --- Recursion ---
def fact(n): return 1 if n<=1 else n * fact(n-1)
print(fact(10))


---
## Summary

| Concept | Key syntax / tool |
|---------|------------------|
| Define | `def name(params): return value` |
| Default arg | `def f(x, y=0)` |
| Variadic | `def f(*args, **kwargs)` |
| Keyword-only | `def f(a, *, b)` — `b` must use keyword |
| Positional-only | `def f(a, /, b)` — `a` cannot use keyword |
| Multiple return | `return a, b` → tuple; unpack: `x, y = f()` |
| Scope | LEGB: Local → Enclosing → Global → Built-in |
| Modify global | `global x` inside function |
| Modify enclosing | `nonlocal x` inside nested function |
| Docstring | First string literal in function body |
| Type hints | `def f(x: int) -> str` |
| Lambda | `lambda x: expr` — one-liner anonymous function |
| map / filter | `map(fn, it)`, `filter(fn, it)` |
| Closure | Inner function + `nonlocal` captures enclosing variables |
| Decorator | `@dec` — wraps function to modify behaviour |
| Decorator factory | `@dec(args)` — decorator that takes arguments |
| functools.wraps | Preserve `__name__`, `__doc__` in decorators |
| functools.lru_cache | Memoize with Least Recently Used cache |
| functools.partial | Pre-fill arguments to create specialised functions |
| functools.reduce | Left-fold over an iterable |
| Recursion | Base case + recursive case; use `@lru_cache` for speedup |

---
*Next up: **Modules & Packages***
